# NB13 — RQ2 Same-Budget Selection, Freeze D-Random and D-Quality

Consumes one pinned frozen **U′** from Notebook 12 and freezes two subsets under one duration budget:

- **D-Random**: uniform over U′, ordered by `SHA256(seed + newline + segment_uid)`.
- **D-Quality**: `quality_score` descending, then `segment_uid` ascending.

Both arms use the same target audio-hour budget. A segment is kept only when `used + duration_seconds <= target`.
If it does not fit, it stays out and the walk continues. Audio is not cut, and rows are not
reordered by length to pack the budget. The realized duration difference between the arms is
allowed only inside the pre-frozen indivisible-segment tolerance (`max_segment_seconds` from
the frozen segmentation contract). A real target must satisfy
`0 < target_seconds <= total U′ duration`. Each arm must select at least one segment.
When the target is smaller than the pool, unused budget must be strictly shorter than every
segment that was left out. Selecting the whole pool with zero unused budget is valid when the
target equals the pool duration.

## 0. Purpose and scientific boundary
- **Input**: the NB12 generation named by `EXPECTED_NB12_GENERATION_ID`. `CURRENT` is audited
  and does not replace that pin. NB11 `SUCCESS_RQ2_U_CLEAN_FROZEN` and NB12
  `SUCCESS_RQ2_PSEUDO_POOL_FROZEN` must both verify.
- **Not in NB13**: teacher inference, pseudo-label regeneration, quality recalibration, G_test,
  test metrics, training, checkpoint choice, or merging into the D0 supervised manifest.
- Overlap between D-Random and D-Quality is recorded and is not an error.
- `SELECTION_BUDGET_HOURS` stays `None` until the thesis budget is fixed. Real selection
  also requires `SELECTION_POLICY_FROZEN` and `RUN_REAL_SELECTION`.

Code lives in `src/rq2_selection_contract.py` and `src/rq2_selection.py`. NB14 will read the frozen manifests.


## 1. Imports and runtime paths

In [ ]:
import json
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir() and (cand / "notebooks").is_dir():
            return cand
    raise RuntimeError("cannot locate project root")

PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rq1_runtime_paths import resolve_rq1_runtime_paths
from src.rq2_selection_contract import (
    SELECTION_RELATIVE_DIR, assert_nb13_output_dir, assert_nb13_sources_clean,
    budget_seconds_from_hours, build_selection_contract, resolve_frozen_u_prime,
)
from src.rq2_selection import (
    publish_same_budget_selection, run_offline_synthetic_self_check, verify_published_selection,
)

RUNTIME = resolve_rq1_runtime_paths(project_root=PROJECT_ROOT)
DURABLE_RQ2_ROOT = RUNTIME.durable_root / "artifacts" / "rq2"
NB11_U_CLEAN_DIR = DURABLE_RQ2_ROOT / "u_clean"
NB12_PSEUDO_DIR = DURABLE_RQ2_ROOT / "pseudo_labels"
OUT_DIR = assert_nb13_output_dir(
    DURABLE_RQ2_ROOT / "selection",
    PROJECT_ROOT,
    durable_root=RUNTIME.durable_root,
)
NB13_SOURCES = [
    PROJECT_ROOT / "src" / "rq2_selection_contract.py",
    PROJECT_ROOT / "src" / "rq2_selection.py",
    PROJECT_ROOT / "notebooks" / "13_RQ2_SameBudget_Selection_Freeze.ipynb",
]
assert_nb13_sources_clean(NB13_SOURCES)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DURABLE_ROOT:", RUNTIME.durable_root)
print("DURABLE_RQ2_ROOT:", DURABLE_RQ2_ROOT)
print("NB11_U_CLEAN_DIR:", NB11_U_CLEAN_DIR)
print("NB12_PSEUDO_DIR:", NB12_PSEUDO_DIR)
print("NB13 output dir:", OUT_DIR)


## 2. Flags

`RUN_REAL_SELECTION` and `SELECTION_POLICY_FROZEN` stay `False` until NB12 has published frozen U′
and this notebook has been reviewed. `SELECTION_BUDGET_HOURS` stays `None` until the shared
audio-duration budget is fixed in hours. `SELECTION_RANDOM_SEED` is the pre-declared D-Random seed;
it is not applied to production U′ while the flags above are off.

In [ ]:
RUN_REAL_SELECTION = False
SELECTION_POLICY_FROZEN = False
SELECTION_BUDGET_HOURS = 20.0
SELECTION_RANDOM_SEED = 42
EXPECTED_NB12_GENERATION_ID = "20261005T095322062456Z-1863e3d9"

print("RUN_REAL_SELECTION:", RUN_REAL_SELECTION)
print("SELECTION_POLICY_FROZEN:", SELECTION_POLICY_FROZEN)
print("SELECTION_BUDGET_HOURS:", SELECTION_BUDGET_HOURS)
print("SELECTION_RANDOM_SEED:", SELECTION_RANDOM_SEED)
print("EXPECTED_NB12_GENERATION_ID:", EXPECTED_NB12_GENERATION_ID)
if RUN_REAL_SELECTION and not SELECTION_POLICY_FROZEN:
    raise RuntimeError("RUN_REAL_SELECTION requires SELECTION_POLICY_FROZEN=True")
if RUN_REAL_SELECTION and SELECTION_BUDGET_HOURS is None:
    raise RuntimeError("RUN_REAL_SELECTION requires an explicit SELECTION_BUDGET_HOURS")


## 3. Resolve the frozen NB12 U′

This cell opens the pinned NB12 generation only when real selection is on.
`CURRENT` is reported by the resolver and cannot replace `EXPECTED_NB12_GENERATION_ID`.


In [ ]:
FROZEN = None
if RUN_REAL_SELECTION:
    FROZEN = resolve_frozen_u_prime(
        PROJECT_ROOT,
        generation_id=EXPECTED_NB12_GENERATION_ID,
        expected_generation_id=EXPECTED_NB12_GENERATION_ID,
        pseudo_dir=NB12_PSEUDO_DIR,
        u_clean_dir=NB11_U_CLEAN_DIR,
        durable_root=RUNTIME.durable_root,
    )
    if FROZEN.identity["resolved_nb12_generation_id"] != EXPECTED_NB12_GENERATION_ID:
        raise RuntimeError("resolved NB12 generation does not match EXPECTED_NB12_GENERATION_ID")
    print("NB12 generation:", FROZEN.generation_id)
    print("NB12 CURRENT audit:", FROZEN.identity["nb12_current_generation_id"])
    print("U′ segments:", FROZEN.identity["n_u_prime"])
    print("U′ ordered uid sha256:", FROZEN.identity["u_prime_ordered_uid_sha256"])
else:
    print("NB12 resolve: skipped (RUN_REAL_SELECTION=False)")
    print("EXPECTED_NB12_GENERATION_ID:", EXPECTED_NB12_GENERATION_ID)


## 4. Selection contract

The contract binds the NB11 hash inherited through NB12, the NB12 contract hash, the U′ file hash,
the ordered U′ UID hash, the hour budget, the random seed, and the three algorithm versions.
It is built only when the budget is an explicit number. No machine path enters the hash.

In [ ]:
SELECTION_CONTRACT = None
if SELECTION_BUDGET_HOURS is None or FROZEN is None:
    print("selection contract: not built (budget is unset or U′ is not loaded)")
else:
    SELECTION_CONTRACT = build_selection_contract(
        FROZEN.identity,
        selection_budget_hours=SELECTION_BUDGET_HOURS,
        random_seed=SELECTION_RANDOM_SEED,
    )
    print("selection_contract_sha256:", SELECTION_CONTRACT["selection_contract_sha256"])
    print("target seconds:", SELECTION_CONTRACT["selection_budget_seconds"])

## 5. Offline synthetic check

Runs an in-memory pool only. It does not read production U′ and does not write `CURRENT`.

In [ ]:
if RUN_REAL_SELECTION:
    print("offline synthetic check: skipped during a real selection run")
else:
    SYNTHETIC = run_offline_synthetic_self_check()
    print(json.dumps(SYNTHETIC, indent=2, sort_keys=True))
    if SYNTHETIC["wrote_current"]:
        raise RuntimeError("synthetic check must not publish CURRENT")

## 6. Real selection

Fail closed unless both freeze flags are on, the budget is set, and `resolve_frozen_u_prime`
has verified the pinned NB12 generation, NB11, the U′ file hash, and the NB12 contract hash.
Publication re-resolves that same pinned U′. It refuses a target larger than the pool, an empty
arm, or a realized-duration gap outside the frozen indivisible-segment tolerance.


In [ ]:
PUBLISHED = None
if not RUN_REAL_SELECTION:
    print("real selection: skipped (RUN_REAL_SELECTION=False)")
else:
    if not SELECTION_POLICY_FROZEN:
        raise RuntimeError("refusing real selection: SELECTION_POLICY_FROZEN is false")
    if SELECTION_BUDGET_HOURS is None:
        raise RuntimeError("refusing real selection: SELECTION_BUDGET_HOURS is None")
    if FROZEN is None:
        raise RuntimeError("refusing real selection: frozen U′ was not resolved")
    budget_seconds_from_hours(SELECTION_BUDGET_HOURS)
    PUBLISHED = publish_same_budget_selection(
        OUT_DIR,
        rows=FROZEN.rows,
        identity=FROZEN.identity,
        selection_budget_hours=SELECTION_BUDGET_HOURS,
        random_seed=SELECTION_RANDOM_SEED,
        policy_frozen=SELECTION_POLICY_FROZEN,
        project_root=PROJECT_ROOT,
        durable_root=RUNTIME.durable_root,
        pseudo_dir=NB12_PSEUDO_DIR,
        u_clean_dir=NB11_U_CLEAN_DIR,
        expected_generation_id=EXPECTED_NB12_GENERATION_ID,
    )
    print("generation:", PUBLISHED["summary"]["generation_id"])
    print("realized gap seconds:", PUBLISHED["summary"]["realized_duration_gap_seconds"])
    print("D-Random:", PUBLISHED["summary"]["d_random"])
    print("D-Quality:", PUBLISHED["summary"]["d_quality"])


## 7. Validation

A successful publication is re-hashed, then both arms are recomputed from the NB12 generation pinned by the selection contract. With the default flags this cell publishes nothing.

In [ ]:
if PUBLISHED is not None:
    checked = verify_published_selection(
        OUT_DIR,
        project_root=PROJECT_ROOT,
        durable_root=RUNTIME.durable_root,
        pseudo_dir=NB12_PSEUDO_DIR,
        u_clean_dir=NB11_U_CLEAN_DIR,
        expected_generation_id=EXPECTED_NB12_GENERATION_ID,
    )
    print("status:", checked["summary"]["status"])
    print("overlap:", checked["summary"]["overlap"])
    print("artifacts:", sorted(checked["generation"]["files"]))
else:
    print("publication: none")
print("RUN_REAL_SELECTION:", RUN_REAL_SELECTION)
print("SELECTION_POLICY_FROZEN:", SELECTION_POLICY_FROZEN)
print("SELECTION_BUDGET_HOURS:", SELECTION_BUDGET_HOURS)
